# 🎯 Module 01: ML Fundamentals

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 01 — ML Fundamentals  
> **Date:** 02 January 2026  
> **Previous:** 00 — Setup and Environment  
> **Next:** 02 — Data Preprocessing

---

Before diving into individual algorithms, we need to understand **how Machine Learning works**.

This notebook covers:

- What Machine Learning is
- Supervised, Unsupervised, and Reinforcement Learning
- Features and targets
- Train / Validation / Test splits
- Bias vs Variance
- Underfitting vs Overfitting
- Generalization
- The complete ML workflow
- The scikit-learn `fit()` / `predict()` API

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what Machine Learning is (and how it differs from traditional programming)
- Distinguish **supervised**, **unsupervised**, and **reinforcement** learning
- Distinguish **regression** from **classification**
- Identify **features** and **targets**
- Split data into **train / validation / test** sets correctly
- Explain the **bias-variance tradeoff**
- Diagnose **underfitting** and **overfitting** from metrics
- Explain **generalization**
- Follow the **end-to-end ML workflow**
- Use scikit-learn's `fit()` / `predict()` API

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.datasets import load_iris, make_regression, make_classification

np.random.seed(42)

print('✅ Setup ready.')

---
## 🧠 1. What is Machine Learning?

**Machine Learning (ML)** is a branch of AI where computers **learn patterns from data** and use those patterns to make predictions or decisions.

### Traditional Programming

```text
Rules + Data → Output
```

Example:

```python
if marks >= 50:
    result = 'Pass'
else:
    result = 'Fail'
```

### Machine Learning

```text
Data + Correct Outputs → Model
Model + New Data → Prediction
```

Example:

```text
House features → ML Model → Predicted Price
```

Instead of manually writing every rule, the model **learns relationships** from examples.

---
## 🗂️ 2. Main Types of Machine Learning

| Type | Data | Goal | Example |
| --- | --- | --- | --- |
| **Supervised** | Labeled | Predict output | House price |
| **Unsupervised** | Unlabeled | Find patterns | Customer groups |
| **Reinforcement** | Rewards | Learn actions | Game-playing agent |

---
### 2.1 Supervised Learning

The dataset contains both **input (X)** and **target (y)**:

| Area | Bedrooms | Price |
| ---: | -------: | ----: |
| 1000 | 2 | 8M |
| 1500 | 3 | 12M |
| 2000 | 4 | 17M |

The model learns:

```text
Area + Bedrooms → Price
```

Two sub-tasks:

- **Regression** → continuous value (price, temperature, sales)
- **Classification** → category (spam/ham, fraud/not, cat/dog)

---
### 2.2 Unsupervised Learning

There is **no target column**:

```text
X → Model → Hidden Patterns
```

```text
Customer data
      ↓
Clustering
      ↓
Group 1 | Group 2 | Group 3
```

Common tasks: clustering, dimensionality reduction, anomaly detection.

Algorithms: K-Means, DBSCAN, PCA (covered later).

---
### 2.3 Reinforcement Learning

An **agent** interacts with an **environment**:

```text
Agent
  ↓ action
Environment
  ↓
Reward / Penalty
  ↓
Agent learns
```

The goal: learn actions that **maximize long-term reward**.

> RL is part of ML, but outside the main classical-ML workflow of this curriculum.

---
## 📊 3. Features and Target

Suppose we want to predict **house prices**:

```text
Area
Bedrooms
Bathrooms
Location
Age
      ↓
   Features (X)
      ↓
    Model
      ↓
Price (y)
```

### Feature
An **input variable** used by the model.

```python
X = [Area, Bedrooms, Bathrooms]
```

### Target
The value we **want to predict**.

```python
y = Price
```

In classification:

```python
X = Email text
y = Spam / Ham
```

---
## ✂️ 4. Train, Validation and Test Data

We split our dataset into **multiple parts**:

```text
Complete Dataset
       │
       ├── Training Data
       ├── Validation Data
       └── Test Data
```

| Dataset | Purpose |
| --- | --- |
| **Training** | Learn patterns |
| **Validation** | Tune / select models |
| **Test** | Final evaluation |

### Common splits

```text
70% Training
15% Validation
15% Testing
```

or:

```text
80% Training
20% Testing
```

with **cross-validation inside the training set**.

### ⚠️ Important rule

> **Never use test data to train or tune your model.**

Otherwise, your final performance estimate becomes unreliable (this is called **data leakage**).

In [ ]:
# Demonstration: correct splitting
X, y = make_regression(n_samples=1000, n_features=5, noise=15, random_state=42)

# Step 1: train vs temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Step 2: split temp → val + test (equal halves)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f'Train:      {len(X_train):>4} samples  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Validation: {len(X_val):>4} samples  ({len(X_val)/len(X)*100:.0f}%)')
print(f'Test:       {len(X_test):>4} samples  ({len(X_test)/len(X)*100:.0f}%)')

---
## ⚖️ 5. Bias vs Variance

A major ML concept is balancing **bias** and **variance**.

### High Bias — Model too simple

```text
Underfitting
```

Performs poorly on **both** training and test data.

### High Variance — Model too complex

```text
Overfitting
```

Performs:
- **Training** → excellent
- **Test** → poor

The model memorized the training data instead of learning general patterns.

### Goal

```text
Good Bias + Good Variance
          ↓
   Better Generalization
```

In [ ]:
# Visualize bias-variance using polynomial fits of different degrees
np.random.seed(42)

# True function: y = sin(x) + noise
x_true = np.linspace(0, 1, 200)
y_true = np.sin(2 * np.pi * x_true)

# Small training sample
x_train = np.random.rand(15)
y_train = np.sin(2 * np.pi * x_train) + 0.15 * np.random.randn(15)

x_plot = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 4, 15]
titles = ['Degree 1 (High Bias / Underfit)',
          'Degree 4 (Good Fit)',
          'Degree 15 (High Variance / Overfit)']

for ax, d, title in zip(axes, degrees, titles):
    coeffs = np.polyfit(x_train, y_train, d)
    y_plot = np.polyval(coeffs, x_plot)
    ax.plot(x_plot, y_true, 'k--', label='true function', alpha=0.6)
    ax.scatter(x_train, y_train, color='red', s=40, zorder=3, label='training points')
    ax.plot(x_plot, y_plot, color='blue', linewidth=2, label=f'degree {d}')
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle('Bias-Variance Tradeoff (polynomial regression)')
plt.tight_layout()
plt.show()

---
## 🚨 6. Underfitting vs Overfitting

| | Underfitting | Good Fit | Overfitting |
| --- | --- | --- | --- |
| Model complexity | Too low | Appropriate | Too high |
| Training error | High | Low | Very low |
| Test error | High | Low | High |
| Generalization | Poor | Good | Poor |

### How to spot them from numbers

**Underfitting:**
```text
Training Accuracy = 60%
Test Accuracy     = 58%
```

**Overfitting:**
```text
Training Accuracy = 99%
Test Accuracy     = 70%
```

**Good fit:**
```text
Training Accuracy = 92%
Test Accuracy     = 90%
```

In [ ]:
# Demonstrating underfitting vs good fit vs overfitting on real data
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=10, n_informative=6,
                           n_redundant=2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def run_model(name, model):
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    gap = train_acc - test_acc
    print(f'{name:25s} | train={train_acc:.3f} | test={test_acc:.3f} | gap={gap:+.3f}')

# Underfit: very shallow tree
run_model('Underfit (depth=1)',  DecisionTreeClassifier(max_depth=1, random_state=42))

# Good fit: moderate depth
run_model('Good fit (depth=4)',  DecisionTreeClassifier(max_depth=4, random_state=42))

# Overfit: unlimited depth
run_model('Overfit (unlimited)', DecisionTreeClassifier(random_state=42))

print('\nNotice: the gap between train and test accuracy tells the story.')

---
## 🎯 7. Generalization

**Generalization** means the model performs well on **new, unseen data**.

One of the **most important goals** of ML:

```text
Training Data
      ↓
    Model
      ↓
Unseen Data
      ↓
Reliable Prediction
```

A model that memorizes training examples but fails on new examples is **not useful** in production.

---
## 🔄 8. The Complete ML Workflow

```text
1. Collect Data
      ↓
2. Explore Data (EDA)
      ↓
3. Clean Data
      ↓
4. Preprocess Data
      ↓
5. Split Data
      ↓
6. Train Model
      ↓
7. Validate / Tune
      ↓
8. Test Model
      ↓
9. Deploy
      ↓
10. Monitor & Improve
```

You'll learn all of these stages throughout this curriculum.

---
## 💻 9. The Scikit-Learn API

Almost every supervised algorithm in scikit-learn follows the same pattern:

```python
model.fit(X_train, y_train)          # learn
predictions = model.predict(X_test)  # use
```

This **consistent API** is one of the biggest reasons scikit-learn is so popular.

In [ ]:
# Regression example (Linear Regression)
np.random.seed(42)
n = 200
area = np.random.randint(500, 3000, n)
bedrooms = np.random.randint(1, 6, n)
noise = np.random.randn(n) * 30
price = 0.5 * area + 20 * bedrooms + 100 + noise

df = pd.DataFrame({'area': area, 'bedrooms': bedrooms, 'price': price})

X = df[['area', 'bedrooms']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Regression:')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}')
print(f'  R²  : {r2_score(y_test, predictions):.3f}')

In [ ]:
# Classification example (Logistic Regression) — same API!
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

print('Classification:')
print(f'  Accuracy: {accuracy_score(y_test, preds):.3f}')
print()
print('Notice: the same fit()/predict() pattern works for both tasks.')

---
## 📝 10. Practice Checklist

Before moving forward, make sure you can explain each of these **in your own words**:

- [ ] What Machine Learning is
- [ ] Supervised vs Unsupervised Learning
- [ ] Regression vs Classification
- [ ] Features vs Target
- [ ] Training vs Validation vs Test data
- [ ] Bias vs Variance
- [ ] Overfitting vs Underfitting
- [ ] Generalization
- [ ] The ML workflow
- [ ] `fit()` vs `predict()`

---
## 🏋️ 11. Hands-On Exercises

### Exercise 1 — Classify the Task
For each scenario, decide whether it's regression or classification:
1. Predicting tomorrow's temperature
2. Predicting if a customer will churn
3. Predicting a house's sale price
4. Predicting if an email is spam
5. Predicting tomorrow's stock price

### Exercise 2 — Split Correctly
Create a synthetic classification dataset with 2000 samples. Split it into **70/15/15** for train/val/test. Verify the counts.

### Exercise 3 — Force Overfitting
Train `DecisionTreeClassifier(max_depth=None)` on a small dataset and compare train accuracy vs test accuracy. Then reduce depth and observe the change.

### Exercise 4 — Force Underfitting
Train a `DecisionTreeClassifier(max_depth=1)` on the same dataset. How different are train and test accuracy?

### Exercise 5 — Same API, Different Task
Import `LinearRegression` and `LogisticRegression`. Confirm that both expose `.fit()` and `.predict()` with the same signature.

### Exercise 6 — Sanity Check
Load the Iris dataset. Split it with `random_state=42`. Then split it with `random_state=1`. Are the splits different? Why does that matter for reproducibility?

---
## 🎤 12. Interview Questions

1. **What is Machine Learning?**  
   A branch of AI where computers learn patterns from data instead of being explicitly programmed.

2. **Difference between AI and ML?**  
   AI is the broad goal of intelligent machines; ML is one approach to achieving it via learning from data.

3. **What is supervised learning?**  
   Learning from **labeled** data (X → y).

4. **Regression vs Classification?**  
   Regression predicts continuous values; classification predicts discrete categories.

5. **What is unsupervised learning?**  
   Learning patterns from **unlabeled** data (no target).

6. **What are features and targets?**  
   Features are the inputs (X); the target is what we want to predict (y).

7. **Why split data into train and test sets?**  
   To evaluate the model on data it has never seen, simulating real-world deployment.

8. **What is overfitting?**  
   When a model performs great on training data but poorly on unseen data.

9. **What is underfitting?**  
   When a model is too simple to capture the underlying pattern (poor on both train and test).

10. **What is the bias-variance tradeoff?**  
    High bias → underfitting; high variance → overfitting. We aim for the balance.

11. **What is generalization?**  
    The ability of a model to perform well on new, unseen data.

12. **Why should test data not be used during training?**  
    It would bias the evaluation and produce unreliable performance estimates.

---
## 🏁 13. Key Takeaways

```text
ML = Learn patterns from data

Supervised     → Labeled data
Unsupervised   → Unlabeled data
Reinforcement  → Rewards/actions

Regression     → Numerical prediction
Classification → Class prediction

Training       → Learn
Validation     → Tune / select
Testing        → Final evaluation

Underfitting   → Model too simple
Overfitting    → Model too complex

Goal           → Generalize to unseen data
```

### 🧭 Golden Rule

> **The goal of ML is not to memorize the training data — it's to generalize to unseen data. Every design decision should serve that goal.**

---

### 🔗 What's Next?

**Module 02 — Data Preprocessing**

Real-world data is messy. We'll learn how to handle missing values, encode categorical variables, scale features, and prepare data for any ML model.